<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/02_del_texto_al_gpt_diminuto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 02 · Del texto al GPT diminuto: tokenizador, entrenamiento desde cero y muestreo

⏱️ **≈ 50 min** (martes 15:00–15:50)

| Parte | Contenido | Tiempo |
|---|---|---|
| A | El tokenizador: de texto a números (BPE) | ≈ 15 min |
| B | Preentrenar un GPT diminuto desde cero (2–4 min de entrenamiento en T4) | ≈ 25 min |
| C | Laboratorio de muestreo: temperatura, top-k, top-p | ≈ 10 min |

Las secciones marcadas con ⭐ **Opcional** se pueden saltar.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

# Parte A · El tokenizador: de texto a números

**Objetivo:** entrenar un tokenizador **BPE** (*Byte Pair Encoding*) con tus textos y ver cómo trocea el lenguaje.

Un ordenador no entiende letras, solo números. El **tokenizador** divide el texto en trozos
(**tokens**) y asigna un número a cada uno. Opciones:

| Unidad | Ejemplo con «caballero» | Problema |
|---|---|---|
| letras | c·a·b·a·l·l·e·r·o | secuencias muy largas |
| palabras | caballero | vocabulario enorme, palabras nuevas desconocidas |
| **subpalabras (BPE)** | caball·ero | ✅ el compromiso que usan GPT, Llama, etc. |

**Cómo funciona BPE:** empieza con caracteres sueltos y une una y otra vez el par más frecuente
(«q»+«u» → «qu», «qu»+«e» → «que»...) hasta alcanzar el tamaño de vocabulario deseado.

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config
cfg = load_config()

CARPETAS = cfg.data.folders
VOCABULARIO = cfg.tokenizer.vocab_size   # prueba 500, 2000, 8000
FRASE = "En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo."

## 1. Cargar textos y entrenar el tokenizador
Tarda unos segundos.

In [ ]:
from dhllm.data import load_texts
from dhllm.tokenizer import train_bpe
textos = load_texts(CARPETAS)
tok = train_bpe(textos, vocab_size=VOCABULARIO, min_frequency=cfg.tokenizer.min_frequency)

## 2. Ver los tokens
Cada color es un token. El espacio suele ir pegado al principio de la palabra.

In [ ]:
from dhllm.tokenizer import show_tokens
show_tokens(tok, FRASE);

Y estos son los números que «ve» el modelo:

In [ ]:
ids = tok.encode(FRASE)
print(ids)
print("De vuelta a texto:", tok.decode(ids))

## 3. ¿Qué ha aprendido el tokenizador?
Los tokens más largos del vocabulario suelen ser palabras frecuentes de tu corpus.

In [ ]:
from dhllm.tokenizer import show_vocabulary_sample
show_vocabulary_sample(tok, n=40);

> ⭐ **Opcional** — puedes saltarte esta sección si vas justo de tiempo.

## 4. Comparar con un tokenizador «de verdad»
Comparamos con el tokenizador de GPT-2 (entrenado sobre todo con inglés) y con el de SmolLM2.
**Menos tokens por palabra = más eficiente** para ese idioma. (Necesita conexión a internet.)

In [ ]:
from dhllm.tokenizer import compare_tokenizers
otros = {"nuestro BPE": tok}
try:
    from transformers import AutoTokenizer
    otros["GPT-2 (inglés)"] = AutoTokenizer.from_pretrained("gpt2")
    otros["SmolLM2"] = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
except Exception as e:
    print("No se pudieron descargar otros tokenizadores:", e)
compare_tokenizers(FRASE, otros);

In [ ]:
if "GPT-2 (inglés)" in otros:
    show_tokens(otros["GPT-2 (inglés)"], FRASE);

## 5. Guardar el tokenizador
Lo usaremos en la parte B.

In [ ]:
tok.save(cfg.tokenizer.save_dir)

## 🖐️ Tu turno
1. Prueba `VOCABULARIO = 300` y `VOCABULARIO = 10000`. ¿Qué cambia en los colores de la frase?
2. Tokeniza una frase en otro idioma (latín, inglés, catalán...) o con ortografía antigua. ¿Qué pasa?
3. **Para pensar:** si un idioma necesita el doble de tokens que el inglés, ¿qué consecuencias tiene
   en el coste y la calidad de los grandes modelos comerciales?

# Parte B · Preentrenar un GPT diminuto desde cero

**Objetivo:** entrenar tu propio modelo de lenguaje (¡con pesos aleatorios al principio!) usando solo tus textos.

La tarea es siempre la misma: **predecir el siguiente token**. El modelo ve un trozo de texto y
tiene que adivinar qué viene después. Al principio adivina al azar; poco a poco mejora.

- La **pérdida** (*loss*) mide cuánto se equivoca. Debe **bajar** durante el entrenamiento.
- **Entrenamiento** vs. **validación**: la validación es texto que el modelo no ve al entrenar.
  Si la pérdida de entrenamiento baja pero la de validación sube, el modelo está **memorizando** (*sobreajuste*).

Nuestro modelo tiene ~2,6 millones de parámetros. GPT-3 tiene 175 000 millones. 🙂
Inspirado en [nanoGPT](https://github.com/karpathy/nanoGPT) y [minimind](https://github.com/jingyaogong/minimind).

⏱️ Con GPU T4: unos 2–4 minutos. En CPU: usa `"max_steps": 300` en los ajustes (≈ 1,5 min, resultados más pobres).

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config, set_seed, get_device
cfg = load_config(overrides={
    "tiny_gpt": {
        # "max_steps": 300,   # ← quita el «#» para entrenar menos pasos (p. ej. en CPU)
        # "n_layer": 6,       # más capas = modelo más grande
    }
})
ajustes = cfg.tiny_gpt
set_seed(cfg.seed)
print("Dispositivo:", get_device())
print(dict(ajustes))
INICIO = "En un lugar de la Mancha"    # texto con el que el modelo empezará a escribir

## 1. Textos y tokenizador
Se carga el tokenizador de la parte A (si no existe, se entrena ahora).

In [ ]:
from pathlib import Path
from dhllm.data import load_texts, join_texts
from dhllm.tokenizer import SimpleTokenizer, train_bpe

textos = load_texts(cfg.data.folders)
ruta_tok = Path(cfg.tokenizer.save_dir) / "tokenizer.json"
if ruta_tok.exists():
    tok = SimpleTokenizer.load(ruta_tok)
    print("Tokenizador cargado:", tok)
else:
    tok = train_bpe(textos, vocab_size=cfg.tokenizer.vocab_size)

## 2. Convertir el corpus en números

In [ ]:
from dhllm.pretrain import encode_corpus
train_ids, val_ids = encode_corpus(tok, join_texts(textos))

## 3. Crear el modelo (vacío)

In [ ]:
from dhllm.pretrain import build_model
modelo = build_model(tok, ajustes)

Antes de entrenar, el modelo escribe sin sentido:

In [ ]:
from dhllm.sampling import generate
print(generate(modelo, tok, INICIO, max_new_tokens=40, seed=0))

## 4. ¡Entrenar!
Verás la pérdida cada cierto número de pasos y, de vez en cuando, una muestra de texto.

In [ ]:
from dhllm.pretrain import train_tiny_gpt, plot_loss
historial = train_tiny_gpt(modelo, tok, train_ids, val_ids, ajustes, sample_prompt=INICIO)
plot_loss(historial)

## 5. ¿Cómo ha evolucionado el texto generado?
**Interpretación de la pérdida:** una pérdida de ~8,3 significa «adivinar al azar» entre 4000 tokens
(ln 4000 ≈ 8,3). Si baja a ~4, el modelo duda en promedio entre unas e⁴ ≈ 55 opciones.

In [ ]:
for paso, texto in historial["samples"]:
    print(f"--- paso {paso} ---\n{texto}\n")

## 6. Generar texto nuevo

In [ ]:
for semilla in range(3):
    print(generate(modelo, tok, INICIO, max_new_tokens=80, temperature=0.8, top_k=50, seed=semilla))
    print("-" * 60)

## 7. Guardar el modelo
Lo usaremos en la parte C. (En Colab, descarga la carpeta `outputs/` si quieres conservarlo.)

In [ ]:
from dhllm.pretrain import save_tiny_gpt
save_tiny_gpt(modelo, tok, ajustes.save_dir)

## 🖐️ Tu turno
1. Cambia `INICIO` por otra frase («Sancho dijo», «Yo por bien tengo»...).

⭐ **Opcional (en casa o si vas rápido; cada prueba vuelve a entrenar):**

2. Entrena con más pasos (`"max_steps": 3000`). ¿Mejora la validación o empieza a subir?
3. Haz el modelo más grande (`"n_layer": 6, "n_embd": 256`) o más pequeño (`"n_layer": 2`). Compara curvas.
4. Entrena **solo** con el Lazarillo: en los ajustes añade `"data": {"folders": ["data/sample/lazarillo_de_tormes.txt"]}` dentro de `overrides`. ¿Qué pasa con tan poco texto?
5. **Para pensar:** ¿«entiende» este modelo el Quijote? ¿Qué ha aprendido exactamente?

# Parte C · Laboratorio de muestreo: temperatura, top-k y top-p

**Objetivo:** entender cómo un modelo **elige** la siguiente palabra.

En cada paso, el modelo calcula una probabilidad para **cada** token del vocabulario. Después hay que
elegir uno:

- **Greedy** (temperatura 0): siempre el más probable → repetitivo.
- **Temperatura**: < 1 hace la distribución más «puntiaguda» (conservador); > 1 la aplana (creativo, caótico).
- **Top-k**: solo se sortea entre los *k* tokens más probables.
- **Top-p** (*nucleus sampling*): solo entre los tokens más probables que suman una probabilidad *p*.

Usamos el GPT diminuto de la parte B (si no existe, se carga SmolLM2-135M).

⏱️ ≈ 10 min. Si vas justo de tiempo, haz solo los puntos 1 y 3.

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config
cfg = load_config()
INICIO = "Dijo don Quijote"
USAR_MODELO = "auto"   # "auto", "tiny_gpt" o un nombre de Hugging Face, p. ej. "HuggingFaceTB/SmolLM2-135M"

In [ ]:
from pathlib import Path
from dhllm.pretrain import load_tiny_gpt
from dhllm.finetune import load_base_model

ruta = Path(cfg.tiny_gpt.save_dir)
if USAR_MODELO in ("auto", "tiny_gpt") and (ruta / "model.pt").exists():
    modelo, tok = load_tiny_gpt(ruta)
    print("Usando el GPT diminuto de la parte B")
else:
    nombre = "HuggingFaceTB/SmolLM2-135M" if USAR_MODELO in ("auto", "tiny_gpt") else USAR_MODELO
    modelo, tok = load_base_model(nombre)

## 1. ¿Qué token viene después?
Estas son las probabilidades del modelo para el siguiente token.

In [ ]:
from dhllm.sampling import next_token_table
next_token_table(modelo, tok, INICIO, temperature=1.0, n=15);

## 2. El efecto de la temperatura
Misma frase, distinta temperatura. Fíjate en cómo cambian las barras.

In [ ]:
for t in [0.3, 1.0, 2.0]:
    next_token_table(modelo, tok, INICIO, temperature=t, n=10)

## 3. Generar con distintos ajustes

In [ ]:
from dhllm.sampling import compare_settings
compare_settings(modelo, tok, INICIO, [
    {"temperature": 0},                                # greedy
    {"temperature": 0.7, "top_k": 50, "top_p": 0.95},  # equilibrado
    {"temperature": 1.5, "top_k": None, "top_p": 1.0}, # caótico
    {"temperature": 1.0, "top_k": 5, "top_p": 1.0},    # solo 5 opciones
], max_new_tokens=cfg.sampling.max_new_tokens // 2);

## 4. Tu propio experimento

In [ ]:
from dhllm.sampling import generate
print(generate(modelo, tok, INICIO,
               max_new_tokens=cfg.sampling.max_new_tokens,
               temperature=0.9,   # ← cambia
               top_k=40,          # ← cambia (None = sin límite)
               top_p=0.9,         # ← cambia (1.0 = sin límite)
               seed=1))

## 🖐️ Tu turno
1. Encuentra los ajustes que producen el texto más «cervantino». ¿Y el más absurdo?
2. Con `temperature=0`, ¿se repite el texto? ¿Por qué?
3. Ejecuta dos veces con `seed=None`. ¿Por qué cambia el resultado?
4. **Para pensar:** un chatbot responde distinto a la misma pregunta. ¿Qué implica esto para citar
   o usar textos generados en investigación?